# 🔬 Optimización de Parámetros TDA para Portfolio Construction

Este notebook realiza una búsqueda exhaustiva de los mejores parámetros de KeplerMapper y DBSCAN para la construcción de portafolios usando Topological Data Analysis (TDA).

**Objetivos:**
1. Probar múltiples combinaciones de parámetros
2. Evaluar performance en entrenamiento y validación
3. Identificar parámetros que generalizan mejor
4. Guardar resultados completos para análisis posterior

**Flujo:**
- Carga de datos (S&P 500 históricos)
- Grid search de parámetros TDA
- Construcción y evaluación de portafolios
- Comparación con benchmark (SPY)
- Análisis de sensibilidad de parámetros

## 📚 1. Librerías e Importaciones

In [1]:
import pandas as pd
import numpy as np
import random
import os
import pickle
from datetime import datetime
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Financial data
import yfinance as yf
import requests
from bs4 import BeautifulSoup

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# TDA and ML
import kmapper as km
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from umap import UMAP

# Portfolio optimization
import cvxpy as cp

print("✅ Librerías importadas correctamente")
print("=" * 80)

✅ Librerías importadas correctamente


## 🎲 2. Configuración de Semillas y Parámetros Globales

In [2]:
# =======================================================
# 🎲 FIJAR SEMILLAS PARA REPRODUCIBILIDAD
# =======================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("🎲 Semillas aleatorias fijadas:")
print(f"   SEED = {SEED}")
print("=" * 80)

# =======================================================
# 📅 CONFIGURACIÓN DE FECHAS
# =======================================================

START_DATE = '2022-01-01'  # Inicio del período completo
END_DATE = '2025-11-01'    # Fin del período completo
AN_DATE = '2024-01-01'     # División entrenamiento/validación

print(f"\n📅 CONFIGURACIÓN DE FECHAS:")
print(f"   Período completo: {START_DATE} a {END_DATE}")
print(f"   Entrenamiento: {START_DATE} a {AN_DATE}")
print(f"   Validación: {AN_DATE} a {END_DATE}")
print("=" * 80)

🎲 Semillas aleatorias fijadas:
   SEED = 42

📅 CONFIGURACIÓN DE FECHAS:
   Período completo: 2022-01-01 a 2025-11-01
   Entrenamiento: 2022-01-01 a 2024-01-01
   Validación: 2024-01-01 a 2025-11-01


## 📊 3. Carga de Datos

### 3.1 Tickers del S&P 500

In [ ]:
# =======================================================
# 📊 CARGAR TICKERS DEL S&P 500
# =======================================================

# Cambiar al directorio FINAL donde están los archivos
final_dir = os.path.join(os.path.dirname(os.getcwd()), 'Optimización_final')
tickers_filename = os.path.join(final_dir, 'tickers_S&P.csv')

if os.path.exists(tickers_filename):
    print(f"✅ Archivo encontrado: {tickers_filename}")
    tickers_df = pd.read_csv(tickers_filename)
    
    if 'Symbol' in tickers_df.columns:
        tickers = tickers_df['Symbol'].tolist()
        tickers = [ticker.replace('.', '-') for ticker in tickers]
        print(f"✅ Tickers cargados: {len(tickers)}")
        print(f"   Primeros 10: {tickers[:10]}")
    else:
        raise ValueError("Columna 'Symbol' no encontrada")
else:
    print(f"❌ Archivo no encontrado: {tickers_filename}")
    print("   Ejecutando descarga desde web...")
    
    url = "https://www.slickcharts.com/sp500"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    table = soup.find('table')
    
    if table:
        df = pd.read_html(str(table))[0]
        tickers = df['Symbol'].tolist()
        tickers = [ticker.replace('.', '-') for ticker in tickers]
        
        # Guardar para uso futuro
        os.makedirs(final_dir, exist_ok=True)
        df.to_csv(tickers_filename, index=False)
        print(f"✅ Tickers descargados y guardados: {len(tickers)}")
    else:
        raise ValueError("No se pudo descargar la lista de tickers")

print("=" * 80)

✅ Archivo encontrado: c:\Users\A01286222\Documents\GitHub\Estancia_Alfredo\FINAL\tickers_S&P.csv
✅ Tickers cargados: 503
   Primeros 10: ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'AVGO', 'GOOG', 'META', 'TSLA', 'BRK-B']


### 3.2 Datos Históricos de Precios

In [4]:
# =======================================================
# 📊 CARGAR/DESCARGAR DATOS HISTÓRICOS
# =======================================================

historicos_filename = os.path.join(final_dir, f'historicos_SP500_{START_DATE}_to_{END_DATE}.csv')

if os.path.exists(historicos_filename):
    print(f"✅ Archivo históricos encontrado")
    print(f"📂 Cargando datos desde '{historicos_filename}'...")
    all_tickers_data = pd.read_csv(historicos_filename, index_col=0, parse_dates=True)
    all_tickers_data.index = pd.to_datetime(all_tickers_data.index)
    print(f"   Datos cargados: {all_tickers_data.shape}")
else:
    print(f"⚠️  Archivo no encontrado")
    print(f"🔍 Descargando datos para {len(tickers)} tickers...")
    print(f"   Esto puede tardar varios minutos...")
    
    all_tickers_data_raw = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
    all_tickers_data = all_tickers_data_raw.dropna(axis=1)
    
    os.makedirs(final_dir, exist_ok=True)
    all_tickers_data.to_csv(historicos_filename)
    print(f"💾 Datos guardados en '{historicos_filename}'")

# Dividir en períodos
tickers_data = all_tickers_data.loc[START_DATE:AN_DATE]
validation_data = all_tickers_data.loc[AN_DATE:END_DATE]

print(f"\n✅ Resumen:")
print(f"   📊 Total: {all_tickers_data.shape}")
print(f"   📊 Entrenamiento: {tickers_data.shape}")
print(f"   📊 Validación: {validation_data.shape}")
print("=" * 80)

✅ Archivo históricos encontrado
📂 Cargando datos desde 'c:\Users\A01286222\Documents\GitHub\Estancia_Alfredo\FINAL\historicos_SP500_2022-01-01_to_2025-11-01.csv'...
   Datos cargados: (962, 495)

✅ Resumen:
   📊 Total: (962, 495)
   📊 Entrenamiento: (501, 495)
   📊 Validación: (461, 495)


### 3.3 Benchmark (SPY)

In [5]:
# =======================================================
# 📊 CARGAR/DESCARGAR SPY (BENCHMARK)
# =======================================================

spy_filename = os.path.join(final_dir, f'historicos_SPY_{START_DATE}_{END_DATE}.csv')

if os.path.exists(spy_filename):
    print(f"✅ Archivo SPY encontrado")
    SPY = pd.read_csv(spy_filename, index_col=0, parse_dates=True)
    SPY.index = pd.to_datetime(SPY.index)
else:
    print(f"⚠️  Archivo SPY no encontrado")
    print("🔍 Descargando datos de SPY...")
    SPY_data = yf.download('SPY', start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
    SPY = SPY_data.to_frame(name='SPY') if isinstance(SPY_data, pd.Series) else SPY_data
    if 'SPY' not in SPY.columns:
        SPY.columns = ['SPY']
    
    os.makedirs(final_dir, exist_ok=True)
    SPY.to_csv(spy_filename)
    print(f"💾 Datos guardados en '{spy_filename}'")

SPY_analysis = SPY.loc[START_DATE:AN_DATE]
SPY_validation = SPY.loc[AN_DATE:END_DATE]

print(f"✅ SPY listo: {SPY.shape}")
print(f"   📊 Entrenamiento: {SPY_analysis.shape}")
print(f"   📊 Validación: {SPY_validation.shape}")
print("=" * 80)

✅ Archivo SPY encontrado
✅ SPY listo: (962, 1)
   📊 Entrenamiento: (501, 1)
   📊 Validación: (461, 1)


### 3.4 Información Fundamental (Market Cap)

In [6]:
# =======================================================
# 📊 CARGAR INFORMACIÓN FUNDAMENTAL
# =======================================================

csv_filename = os.path.join(final_dir, 'sp500_ticker_info_database.csv')

if os.path.exists(csv_filename):
    print(f"✅ Archivo de info fundamental encontrado")
    ticker_info_df = pd.read_csv(csv_filename, index_col=0)
    ticker_info_db = ticker_info_df.to_dict('index')
    print(f"   📊 Total de tickers: {len(ticker_info_df)}")
else:
    print(f"⚠️  Archivo no encontrado: {csv_filename}")
    print("   Este archivo es necesario para el análisis")
    print("   Ejecuta primero 'Mapper_500_Creación.ipynb' para generarlo")
    raise FileNotFoundError(f"'{csv_filename}' no encontrado")

# Verificar market caps
valid_mcaps = {t: info['market_cap'] for t, info in ticker_info_db.items() 
               if info.get('market_cap') is not None}

print(f"   ✅ Market caps válidos: {len(valid_mcaps)}")
print("=" * 80)

✅ Archivo de info fundamental encontrado
   📊 Total de tickers: 503
   ✅ Market caps válidos: 503


## 🧬 4. Preparación de Datos para TDA

In [7]:
# =======================================================
# 📊 PREPARAR DATOS PARA TDA
# =======================================================

print("📊 PREPARACIÓN DE DATOS PARA TDA")
print("=" * 80)

# Seleccionar universo disponible
market_caps = {ticker: info['market_cap'] 
               for ticker, info in ticker_info_db.items() 
               if info.get('market_cap') is not None}

available_tickers = [t for t in tickers_data.columns if t in market_caps]
universe_data = tickers_data[available_tickers].copy()

print(f"✅ Universo: {len(available_tickers)} tickers")
print(f"📊 Datos: {universe_data.shape}")

# Calcular log-returns
log_returns = np.log(universe_data / universe_data.shift(1)).dropna()
log_returns_T = log_returns.T.values  # Transponer: cada fila es un ticker

print(f"✅ Log-returns calculados: {log_returns.shape}")
print(f"   Transpuesto para TDA: {log_returns_T.shape} (tickers x días)")
print("=" * 80)

📊 PREPARACIÓN DE DATOS PARA TDA
✅ Universo: 495 tickers
📊 Datos: (501, 495)
✅ Log-returns calculados: (500, 495)
   Transpuesto para TDA: (495, 500) (tickers x días)


## ⚙️ 5. Definición del Grid de Parámetros

In [22]:
# =======================================================
# ⚙️ GRID DE PARÁMETROS PARA BÚSQUEDA
# =======================================================

print("⚙️ DEFINICIÓN DEL GRID DE PARÁMETROS")
print("=" * 80)

# Grid de búsqueda (ajustar según necesites más/menos iteraciones)
param_grid = {
    # Reducción de dimensionalidad
    'pca_variance': [0.5,  0.7, 0.9],
    'umap_dim': [1, 2],
    'umap_neighbors': [10, 15, 20],
    'umap_min_dist': [ 0.1, 0.2],
    
    # Cover (cobertura del espacio)
    'n_cubes': [4, 6, 8, 10],
    'perc_overlap': [0.3, 0.4, 0.5],
    
    # Clustering DBSCAN
    #'dbscan_eps': [0.3, 0.4, 0.5],
    'min_samples': [4]
}

# Calcular número total de combinaciones
total_combinations = np.prod([len(v) for v in param_grid.values()])

print(f"📊 Parámetros a probar:")
for param, values in param_grid.items():
    print(f"   • {param}: {values}")

print(f"\n🔢 Total de combinaciones posibles: {total_combinations:,}")
print(f"⚠️  ADVERTENCIA: Esto puede tardar varias horas")
print("=" * 80)

# Preguntar al usuario si quiere usar grid reducido
USE_REDUCED_GRID = False  # Cambiar a False para grid completo

if USE_REDUCED_GRID:
    print("\n🎯 USANDO GRID REDUCIDO (más rápido)")
    param_grid_reduced = {
        'pca_variance': [0.5, 0.6, 0.7],
        'umap_dim': [1, 2],
        'umap_neighbors': [15],
        'umap_min_dist': [0.1],
        'n_cubes': [5, 6, 8],
        'perc_overlap': [0.3, 0.4],
        #'dbscan_eps': [0.3, 0.35, 0.4],
        'min_samples': [3]
    }
    param_grid = param_grid_reduced
    total_combinations = np.prod([len(v) for v in param_grid.values()])
    print(f"   Total de combinaciones: {total_combinations}")
    print("=" * 80)

⚙️ DEFINICIÓN DEL GRID DE PARÁMETROS
📊 Parámetros a probar:
   • pca_variance: [0.5, 0.7, 0.9]
   • umap_dim: [1, 2]
   • umap_neighbors: [10, 15, 20]
   • umap_min_dist: [0.1, 0.2]
   • n_cubes: [4, 6, 8, 10]
   • perc_overlap: [0.3, 0.4, 0.5]
   • min_samples: [4]

🔢 Total de combinaciones posibles: 432
⚠️  ADVERTENCIA: Esto puede tardar varias horas


## 🔬 6. Funciones de Evaluación

In [23]:
# =======================================================
# 📊 FUNCIONES DE EVALUACIÓN DE PORTAFOLIOS
# =======================================================

def calculate_portfolio_metrics(returns, portfolio_returns, name="Portfolio"):
    """
    Calcula métricas de performance de un portafolio
    """
    # Retornos anualizados
    ann_return = portfolio_returns.mean() * 252
    
    # Volatilidad anualizada
    ann_vol = portfolio_returns.std() * np.sqrt(252)
    
    # Sharpe Ratio (asumiendo risk-free = 0)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Max Drawdown
    cum_returns = (1 + portfolio_returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = (cum_returns - running_max) / running_max
    max_dd = drawdown.min()
    
    # Retorno total
    total_return = (1 + portfolio_returns).prod() - 1
    
    return {
        'name': name,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'total_return': total_return,
        'num_periods': len(portfolio_returns)
    }

def build_mapper_portfolio(params, log_returns_T, available_tickers, verbose=False):
    """
    Construye un portafolio usando TDA con los parámetros dados
    
    Returns:
        graph, node_clusters, projected_data, mapper
    """
    try:
        # Inicializar mapper
        mapper = km.KeplerMapper(verbose=0)
        
        # Proyección con PCA + UMAP
        projected_data = mapper.fit_transform(
            log_returns_T,
            projection=[
                PCA(n_components=params['pca_variance'], random_state=SEED),
                UMAP(n_components=params['umap_dim'], 
                     random_state=SEED, 
                     n_jobs=1,  # Determinístico
                     n_neighbors=params['umap_neighbors'],
                     min_dist=params['umap_min_dist'])
            ]
        )
        
        # Construir grafo con DBSCAN
        graph = mapper.map(
            projected_data,
            log_returns_T,
            clusterer=DBSCAN(
                metric='correlation',
                #eps=params['dbscan_eps'],
                min_samples=params['min_samples'],
                n_jobs=1
            ),
            cover=km.Cover(
                n_cubes=params['n_cubes'],
                perc_overlap=params['perc_overlap']
            )
        )
        
        # Extraer clusters
        node_clusters = {}
        for node_id, node_members in graph['nodes'].items():
            node_clusters[node_id] = [available_tickers[i] for i in node_members]
        
        if verbose:
            print(f"   ✅ Grafo: {len(graph['nodes'])} nodos, {len(graph['links'])} enlaces")
            cluster_sizes = [len(m) for m in node_clusters.values()]
            print(f"   📊 Clusters: {len(node_clusters)} (tamaño promedio: {np.mean(cluster_sizes):.1f})")
        
        return graph, node_clusters, projected_data, mapper
        
    except Exception as e:
        if verbose:
            print(f"   ❌ Error construyendo mapper: {str(e)}")
        return None, None, None, None

def optimize_cluster_portfolio(cluster_tickers, returns_data, market_caps, method='equal'):
    """
    Optimiza pesos dentro de un cluster
    
    methods:
        - 'equal': Equal weight
        - 'market_cap': Market cap weighted
        - 'min_vol': Minimum volatility
        - 'max_sharpe': Maximum Sharpe ratio
    """
    cluster_returns = returns_data[cluster_tickers]
    
    if method == 'equal':
        weights = np.ones(len(cluster_tickers)) / len(cluster_tickers)
        
    elif method == 'market_cap':
        mcaps = np.array([market_caps.get(t, 1) for t in cluster_tickers])
        weights = mcaps / mcaps.sum()
        
    elif method == 'min_vol':
        try:
            cov_matrix = cluster_returns.cov()
            n = len(cluster_tickers)
            w = cp.Variable(n)
            risk = cp.quad_form(w, cov_matrix.values)
            constraints = [cp.sum(w) == 1, w >= 0]
            prob = cp.Problem(cp.Minimize(risk), constraints)
            prob.solve()
            weights = w.value if prob.status == 'optimal' else np.ones(n) / n
        except:
            weights = np.ones(len(cluster_tickers)) / len(cluster_tickers)
            
    elif method == 'max_sharpe':
        try:
            mean_returns = cluster_returns.mean()
            cov_matrix = cluster_returns.cov()
            n = len(cluster_tickers)
            w = cp.Variable(n)
            ret = mean_returns.values @ w
            risk = cp.quad_form(w, cov_matrix.values)
            constraints = [cp.sum(w) == 1, w >= 0]
            prob = cp.Problem(cp.Maximize(ret / cp.sqrt(risk)), constraints)
            prob.solve()
            weights = w.value if prob.status == 'optimal' else np.ones(n) / n
        except:
            weights = np.ones(len(cluster_tickers)) / len(cluster_tickers)
    
    else:
        weights = np.ones(len(cluster_tickers)) / len(cluster_tickers)
    
    # Normalizar por si acaso
    weights = np.array(weights)
    weights = weights / weights.sum()
    
    return dict(zip(cluster_tickers, weights))

print("✅ Funciones de evaluación definidas")
print("=" * 80)

✅ Funciones de evaluación definidas


## 🚀 7. Búsqueda de Parámetros Óptimos

In [24]:
# =======================================================
# 🚀 GRID SEARCH DE PARÁMETROS
# =======================================================

print("🚀 INICIANDO GRID SEARCH DE PARÁMETROS")
print("=" * 80)

# Crear todas las combinaciones
param_names = list(param_grid.keys())
param_values = list(param_grid.values())
param_combinations = list(product(*param_values))

print(f"📊 Total de combinaciones a probar: {len(param_combinations)}")
print(f"⏱️  Tiempo estimado: ~{len(param_combinations) * 10 / 60:.1f} minutos")
print("=" * 80)

# Almacenar resultados
results = []
failed_combinations = []

# Calcular returns de validación una sola vez
validation_returns = np.log(validation_data / validation_data.shift(1)).dropna()

# Iterar sobre combinaciones
for idx, param_combo in enumerate(param_combinations, 1):
    # Crear diccionario de parámetros
    params = dict(zip(param_names, param_combo))
    
    if idx % 10 == 0 or idx == 1:
        print(f"\n📍 Progreso: {idx}/{len(param_combinations)} ({idx/len(param_combinations)*100:.1f}%)")
        print(f"   Parámetros actuales:")
        for k, v in params.items():
            print(f"      {k}: {v}")
    
    try:
        # Construir mapper
        graph, node_clusters, projected_data, mapper_obj = build_mapper_portfolio(
            params, log_returns_T, available_tickers, verbose=(idx % 20 == 0)
        )
        
        if graph is None or len(node_clusters) == 0:
            failed_combinations.append((params, "No clusters generated"))
            continue
        
        # Seleccionar top clusters por Sharpe ratio
        cluster_metrics = {}
        for cluster_id, cluster_tickers in node_clusters.items():
            if len(cluster_tickers) < 2:
                continue
            
            cluster_returns = log_returns[cluster_tickers].mean(axis=1)
            cluster_sharpe = (cluster_returns.mean() * 252) / (cluster_returns.std() * np.sqrt(252))
            cluster_metrics[cluster_id] = {
                'sharpe': cluster_sharpe,
                'tickers': cluster_tickers,
                'size': len(cluster_tickers)
            }
        
        # Seleccionar top 3 clusters
        top_clusters = sorted(cluster_metrics.items(), 
                            key=lambda x: x[1]['sharpe'], 
                            reverse=True)[:3]
        
        if len(top_clusters) == 0:
            failed_combinations.append((params, "No valid clusters"))
            continue
        
        # Construir portafolio combinando top clusters
        portfolio_weights = {}
        cluster_weight = 1.0 / len(top_clusters)
        
        for cluster_id, cluster_info in top_clusters:
            cluster_tickers = cluster_info['tickers']
            
            # Optimizar dentro del cluster (market cap weighted)
            intra_weights = optimize_cluster_portfolio(
                cluster_tickers, log_returns, valid_mcaps, method='market_cap'
            )
            
            # Agregar al portafolio global
            for ticker, weight in intra_weights.items():
                portfolio_weights[ticker] = portfolio_weights.get(ticker, 0) + weight * cluster_weight
        
        # Normalizar pesos
        total_weight = sum(portfolio_weights.values())
        portfolio_weights = {t: w/total_weight for t, w in portfolio_weights.items()}
        
        # Calcular returns del portafolio en entrenamiento
        port_tickers = list(portfolio_weights.keys())
        port_w = np.array([portfolio_weights[t] for t in port_tickers])
        train_returns = (log_returns[port_tickers] * port_w).sum(axis=1)
        
        # Calcular metrics en entrenamiento
        train_metrics = calculate_portfolio_metrics(log_returns, train_returns, "Train")
        
        # Calcular returns en validación
        val_tickers_available = [t for t in port_tickers if t in validation_returns.columns]
        if len(val_tickers_available) < len(port_tickers) * 0.8:  # Al menos 80% disponibles
            failed_combinations.append((params, "Too many tickers missing in validation"))
            continue
        
        val_weights_adjusted = {t: portfolio_weights[t] for t in val_tickers_available}
        total_w = sum(val_weights_adjusted.values())
        val_weights_adjusted = {t: w/total_w for t, w in val_weights_adjusted.items()}
        
        val_w = np.array([val_weights_adjusted[t] for t in val_tickers_available])
        val_returns = (validation_returns[val_tickers_available] * val_w).sum(axis=1)
        
        # Calcular metrics en validación
        val_metrics = calculate_portfolio_metrics(validation_returns, val_returns, "Validation")
        
        # Guardar resultados
        result = {
            **params,
            'num_nodes': len(graph['nodes']),
            'num_links': len(graph['links']),
            'num_clusters': len(node_clusters),
            'num_clusters_used': len(top_clusters),
            'num_tickers_portfolio': len(portfolio_weights),
            'train_sharpe': train_metrics['sharpe'],
            'train_return': train_metrics['ann_return'],
            'train_vol': train_metrics['ann_vol'],
            'train_max_dd': train_metrics['max_dd'],
            'val_sharpe': val_metrics['sharpe'],
            'val_return': val_metrics['ann_return'],
            'val_vol': val_metrics['ann_vol'],
            'val_max_dd': val_metrics['max_dd'],
            'sharpe_diff': train_metrics['sharpe'] - val_metrics['sharpe'],
            'overfitting_score': abs(train_metrics['sharpe'] - val_metrics['sharpe']) / train_metrics['sharpe'] if train_metrics['sharpe'] > 0 else 999,
            'success': True
        }
        
        results.append(result)
        
    except Exception as e:
        failed_combinations.append((params, str(e)))
        if idx % 20 == 0:
            print(f"   ❌ Error: {str(e)[:80]}")

print(f"\n{'='*80}")
print(f"✅ GRID SEARCH COMPLETADO")
print(f"   Total probadas: {len(param_combinations)}")
print(f"   Exitosas: {len(results)}")
print(f"   Fallidas: {len(failed_combinations)}")
print("=" * 80)

🚀 INICIANDO GRID SEARCH DE PARÁMETROS
📊 Total de combinaciones a probar: 432
⏱️  Tiempo estimado: ~72.0 minutos

📍 Progreso: 1/432 (0.2%)
   Parámetros actuales:
      pca_variance: 0.5
      umap_dim: 1
      umap_neighbors: 10
      umap_min_dist: 0.1
      n_cubes: 4
      perc_overlap: 0.3
      min_samples: 4

📍 Progreso: 10/432 (2.3%)
   Parámetros actuales:
      pca_variance: 0.5
      umap_dim: 1
      umap_neighbors: 10
      umap_min_dist: 0.1
      n_cubes: 10
      perc_overlap: 0.3
      min_samples: 4

📍 Progreso: 10/432 (2.3%)
   Parámetros actuales:
      pca_variance: 0.5
      umap_dim: 1
      umap_neighbors: 10
      umap_min_dist: 0.1
      n_cubes: 10
      perc_overlap: 0.3
      min_samples: 4

📍 Progreso: 20/432 (4.6%)
   Parámetros actuales:
      pca_variance: 0.5
      umap_dim: 1
      umap_neighbors: 10
      umap_min_dist: 0.2
      n_cubes: 8
      perc_overlap: 0.4
      min_samples: 4

📍 Progreso: 20/432 (4.6%)
   Parámetros actuales:
      pca_varian

## 💾 8. Guardar Resultados

In [25]:
# =======================================================
# 💾 GUARDAR RESULTADOS COMPLETOS
# =======================================================

print("💾 GUARDANDO RESULTADOS")
print("=" * 80)

# Crear DataFrame con resultados
results_df = pd.DataFrame(results)

# Timestamp para el archivo
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Guardar CSV
csv_output = f'optimization_results_{timestamp}.csv'
results_df.to_csv(csv_output, index=False)
print(f"✅ Resultados guardados en CSV: {csv_output}")

# Guardar pickle con todos los datos
pickle_output = f'optimization_full_{timestamp}.pkl'
full_data = {
    'results_df': results_df,
    'param_grid': param_grid,
    'failed_combinations': failed_combinations,
    'config': {
        'START_DATE': START_DATE,
        'END_DATE': END_DATE,
        'AN_DATE': AN_DATE,
        'SEED': SEED,
        'total_combinations': len(param_combinations),
        'successful': len(results),
        'failed': len(failed_combinations)
    }
}

with open(pickle_output, 'wb') as f:
    pickle.dump(full_data, f)

print(f"✅ Datos completos guardados en pickle: {pickle_output}")
print(f"\n📊 Resumen:")
print(f"   Total de experimentos: {len(results)}")
print(f"   Columnas en resultados: {len(results_df.columns)}")
print("=" * 80)

💾 GUARDANDO RESULTADOS
✅ Resultados guardados en CSV: optimization_results_20251201_165618.csv
✅ Datos completos guardados en pickle: optimization_full_20251201_165618.pkl

📊 Resumen:
   Total de experimentos: 432
   Columnas en resultados: 23


## 📊 9. Análisis de Resultados

### 9.1 Top Configuraciones por Sharpe en Validación

In [ ]:
# =======================================================
# 📊 TOP CONFIGURACIONES POR SHARPE EN VALIDACIÓN
# =======================================================

print("📊 TOP 10 CONFIGURACIONES POR SHARPE EN VALIDACIÓN")
print("=" * 80)

# Ordenar por Sharpe en validación
top_configs = results_df.nlargest(10, 'val_sharpe')

print("\n🏆 Top 10 por Sharpe en Validación:\n")
for idx, row in top_configs.iterrows():
    print(f"{idx+1}. Sharpe Val: {row['val_sharpe']:.3f} | Train: {row['train_sharpe']:.3f} | Diff: {row['sharpe_diff']:.3f}")
    print(f"   PCA: {row['pca_variance']:.1f} | UMAP: {row['umap_dim']}D | Cubos: {row['n_cubes']} ")#| eps: {row['dbscan_eps']}
    print(f"   Clusters: {row['num_clusters']} | Portfolio: {row['num_tickers_portfolio']} tickers")
    print()

# Mostrar DataFrame completo de top 10
display(top_configs[[
    'val_sharpe', 'train_sharpe', 'sharpe_diff', 'overfitting_score',
    'pca_variance', 'umap_dim', 'n_cubes', 'perc_overlap', 
     'min_samples', 'num_clusters', 'num_tickers_portfolio'#'dbscan_eps',
]])

print("=" * 80)

📊 TOP 10 CONFIGURACIONES POR SHARPE EN VALIDACIÓN

🏆 Top 10 por Sharpe en Validación:

309. Sharpe Val: 1.521 | Train: 0.431 | Diff: -1.090
   PCA: 0.9 | UMAP: 1D | Cubos: 8 
   Clusters: 14 | Portfolio: 25 tickers

106. Sharpe Val: 1.445 | Train: 0.779 | Diff: -0.665
   PCA: 0.5 | UMAP: 2D | Cubos: 10 
   Clusters: 40 | Portfolio: 16 tickers

312. Sharpe Val: 1.396 | Train: 0.482 | Diff: -0.914
   PCA: 0.9 | UMAP: 1D | Cubos: 10 
   Clusters: 16 | Portfolio: 38 tickers

310. Sharpe Val: 1.321 | Train: 0.549 | Diff: -0.772
   PCA: 0.9 | UMAP: 1D | Cubos: 10 
   Clusters: 16 | Portfolio: 23 tickers

290. Sharpe Val: 1.254 | Train: 0.071 | Diff: -1.183
   PCA: 0.9 | UMAP: 1D | Cubos: 4 
   Clusters: 5 | Portfolio: 358 tickers

414. Sharpe Val: 1.241 | Train: 0.428 | Diff: -0.813
   PCA: 0.9 | UMAP: 2D | Cubos: 6 
   Clusters: 45 | Portfolio: 76 tickers

311. Sharpe Val: 1.197 | Train: 0.608 | Diff: -0.590
   PCA: 0.9 | UMAP: 1D | Cubos: 10 
   Clusters: 17 | Portfolio: 24 tickers

306. S

,val_sharpe,train_sharpe,sharpe_diff,overfitting_score,pca_variance,umap_dim,n_cubes,perc_overlap,min_samples,num_clusters,num_tickers_portfolio
308,1.520872,0.430945,-1.089927,2.529153,0.9,1,8,0.5,4,14,25
105,1.444843,0.779347,-0.665496,0.853914,0.5,2,10,0.3,4,40,16
311,1.395820,0.482277,-0.913543,1.894229,0.9,1,10,0.5,4,16,38
309,1.320838,0.548546,-0.772292,1.407889,0.9,1,10,0.3,4,16,23
289,1.253788,0.070554,-1.183234,16.770601,0.9,1,4,0.4,4,5,358
413,1.241376,0.428023,-0.813353,1.900255,0.9,2,6,0.5,4,45,76
310,1.197250,0.607504,-0.589746,0.970768,0.9,1,10,0.4,4,17,24
305,1.141041,0.481550,-0.659491,1.369517,0.9,1,6,0.5,4,11,130
385,1.134720,0.306256,-0.828464,2.705134,0.9,2,4,0.4,4,19,56
303,1.134306,0.481088,-0.653218,1.357794,0.9,1,6,0.3,4,10,97


: 

### 9.2 Mejor Configuración (Menor Overfitting)

In [ ]:
# =======================================================
# 🎯 MEJOR CONFIGURACIÓN (MENOR OVERFITTING)
# =======================================================

print("🎯 MEJOR CONFIGURACIÓN (MENOR OVERFITTING)")
print("=" * 80)

# Filtrar solo configuraciones con Sharpe positivo en validación
valid_configs = results_df[results_df['val_sharpe'] > 0].copy()

# Ordenar por overfitting score (menor es mejor)
best_config = valid_configs.nsmallest(1, 'overfitting_score').iloc[0]

print("\n🏆 CONFIGURACIÓN ÓPTIMA (Mejor Generalización):\n")
print(f"📊 Performance:")
print(f"   Sharpe Train: {best_config['train_sharpe']:.3f}")
print(f"   Sharpe Val: {best_config['val_sharpe']:.3f}")
print(f"   Diferencia: {best_config['sharpe_diff']:.3f}")
print(f"   Overfitting Score: {best_config['overfitting_score']:.3f}")

print(f"\n⚙️ Parámetros:")
print(f"   PCA Variance: {best_config['pca_variance']}")
print(f"   UMAP Dim: {best_config['umap_dim']}")
print(f"   UMAP Neighbors: {best_config['umap_neighbors']}")
print(f"   UMAP Min Dist: {best_config['umap_min_dist']}")
print(f"   N Cubes: {best_config['n_cubes']}")
print(f"   Overlap: {best_config['perc_overlap']}")
print(f"   DBSCAN eps: {best_config['dbscan_eps']}")
print(f"   Min Samples: {best_config['min_samples']}")

print(f"\n📈 Estructura:")
print(f"   Nodos: {best_config['num_nodes']}")
print(f"   Enlaces: {best_config['num_links']}")
print(f"   Clusters: {best_config['num_clusters']}")
print(f"   Tickers en Portfolio: {best_config['num_tickers_portfolio']}")

print("=" * 80)

🎯 MEJOR CONFIGURACIÓN (MENOR OVERFITTING)

🏆 CONFIGURACIÓN ÓPTIMA (Mejor Generalización):

📊 Performance:
   Sharpe Train: 1.475
   Sharpe Val: 1.468
   Diferencia: 0.007
   Overfitting Score: 0.005

⚙️ Parámetros:
   PCA Variance: 0.5
   UMAP Dim: 1
   UMAP Neighbors: 15
   UMAP Min Dist: 0.1
   N Cubes: 6
   Overlap: 0.4
   DBSCAN eps: 0.4
   Min Samples: 3

📈 Estructura:
   Nodos: 14
   Enlaces: 7
   Clusters: 14
   Tickers en Portfolio: 18


### 9.3 Análisis de Sensibilidad de Parámetros

In [ ]:
# =======================================================
# 📊 ANÁLISIS DE SENSIBILIDAD DE PARÁMETROS
# =======================================================

print("📊 ANÁLISIS DE SENSIBILIDAD DE PARÁMETROS")
print("=" * 80)

param_importance = {}

for param in param_names:
    # Calcular correlación entre parámetro y Sharpe en validación
    if param in ['pca_variance', 'umap_dim', 'umap_neighbors', 'umap_min_dist',
                 'n_cubes', 'perc_overlap', 'dbscan_eps', 'min_samples']:
        corr = results_df[param].corr(results_df['val_sharpe'])
        param_importance[param] = corr

# Ordenar por importancia absoluta
sorted_importance = sorted(param_importance.items(), 
                          key=lambda x: abs(x[1]), 
                          reverse=True)

print("\n📈 IMPORTANCIA DE PARÁMETROS (Correlación con Sharpe Validación):\n")
for param, corr in sorted_importance:
    direction = "📈 Positiva" if corr > 0 else "📉 Negativa"
    impact = "🔥 Alta" if abs(corr) > 0.3 else "⚡ Media" if abs(corr) > 0.1 else "💤 Baja"
    print(f"{impact} {param:20s}: {corr:+.3f} {direction}")

print("\n💡 INTERPRETACIÓN:")
print("   • Correlación > 0.3: Parámetro muy importante")
print("   • Correlación 0.1-0.3: Parámetro moderadamente importante")
print("   • Correlación < 0.1: Parámetro poco importante")
print("=" * 80)

📊 ANÁLISIS DE SENSIBILIDAD DE PARÁMETROS

📈 IMPORTANCIA DE PARÁMETROS (Correlación con Sharpe Validación):

⚡ Media pca_variance        : -0.116 📉 Negativa
💤 Baja umap_dim            : +0.096 📈 Positiva
💤 Baja umap_neighbors      : +nan 📉 Negativa
🔥 Alta dbscan_eps          : +0.728 📈 Positiva
💤 Baja n_cubes             : -0.048 📉 Negativa
💤 Baja perc_overlap        : +0.047 📈 Positiva
💤 Baja umap_min_dist       : +0.000 📈 Positiva
💤 Baja min_samples         : +nan 📉 Negativa

💡 INTERPRETACIÓN:
   • Correlación > 0.3: Parámetro muy importante
   • Correlación 0.1-0.3: Parámetro moderadamente importante
   • Correlación < 0.1: Parámetro poco importante


### 9.4 Visualizaciones

In [ ]:
# =======================================================
# 📊 VISUALIZACIONES DE RESULTADOS
# =======================================================

print("📊 GENERANDO VISUALIZACIONES")
print("=" * 80)

# 1. Scatter: Train vs Validation Sharpe
fig1 = px.scatter(
    results_df,
    x='train_sharpe',
    y='val_sharpe',
    color='overfitting_score',
    size='num_tickers_portfolio',
    hover_data=['pca_variance', 'umap_dim', 'n_cubes', 'dbscan_eps'],
    title='Train vs Validation Sharpe Ratio',
    labels={'train_sharpe': 'Train Sharpe', 'val_sharpe': 'Validation Sharpe'},
    color_continuous_scale='RdYlGn_r'
)
fig1.add_shape(type='line', x0=0, x1=results_df['train_sharpe'].max(),
               y0=0, y1=results_df['train_sharpe'].max(),
               line=dict(dash='dash', color='gray'))
fig1.show()

# 2. Box plots por parámetro
fig2 = make_subplots(
    rows=2, cols=4,
    subplot_titles=['PCA Variance', 'UMAP Dim', 'N Cubes', 'Overlap',
                   'DBSCAN eps', 'Min Samples', 'UMAP Neighbors', 'UMAP Min Dist']
)

params_to_plot = ['pca_variance', 'umap_dim', 'n_cubes', 'perc_overlap',
                  'dbscan_eps', 'min_samples', 'umap_neighbors', 'umap_min_dist']

for idx, param in enumerate(params_to_plot, 1):
    row = (idx - 1) // 4 + 1
    col = (idx - 1) % 4 + 1
    
    for value in results_df[param].unique():
        subset = results_df[results_df[param] == value]
        fig2.add_trace(
            go.Box(y=subset['val_sharpe'], name=str(value), showlegend=False),
            row=row, col=col
        )

fig2.update_layout(height=800, title_text="Distribución de Sharpe Validación por Parámetro")
fig2.show()

# 3. Heatmap de correlaciones
corr_matrix = results_df[[
    'pca_variance', 'umap_dim', 'n_cubes', 'perc_overlap',
    'dbscan_eps', 'min_samples', 'val_sharpe', 'overfitting_score'
]].corr()

fig3 = px.imshow(
    corr_matrix,
    title='Correlación entre Parámetros y Métricas',
    color_continuous_scale='RdBu_r',
    aspect='auto'
)
fig3.show()

print("✅ Visualizaciones generadas")
print("=" * 80)

📊 GENERANDO VISUALIZACIONES


✅ Visualizaciones generadas


## 🎯 10. Construcción del Portfolio Óptimo Final

In [ ]:
# =======================================================
# 🎯 CONSTRUIR PORTFOLIO CON MEJOR CONFIGURACIÓN
# =======================================================

print("🎯 CONSTRUYENDO PORTFOLIO CON CONFIGURACIÓN ÓPTIMA")
print("=" * 80)

# Usar la mejor configuración
best_params = {
    'pca_variance': best_config['pca_variance'],
    'umap_dim': int(best_config['umap_dim']),
    'umap_neighbors': int(best_config['umap_neighbors']),
    'umap_min_dist': best_config['umap_min_dist'],
    'n_cubes': int(best_config['n_cubes']),
    'perc_overlap': best_config['perc_overlap'],
    'dbscan_eps': best_config['dbscan_eps'],
    'min_samples': int(best_config['min_samples'])
}

print("⚙️ Parámetros óptimos:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# Construir mapper final
print("\n🔨 Construyendo mapper final...")
final_graph, final_clusters, final_projection, final_mapper = build_mapper_portfolio(
    best_params, log_returns_T, available_tickers, verbose=True
)

# Guardar visualización HTML
html_filename = f'optimal_mapper_{timestamp}.html'
html_output = final_mapper.visualize(
    final_graph,
    path_html=html_filename,
    title=f"Optimal Portfolio Mapper (Sharpe Val: {best_config['val_sharpe']:.3f})",
    custom_tooltips=np.array([f"{ticker}" for ticker in available_tickers])
)

print(f"\n✅ Visualización guardada: {html_filename}")

# Guardar configuración óptima
optimal_config = {
    'params': best_params,
    'graph': final_graph,
    'clusters': final_clusters,
    'projection': final_projection,
    'metrics': {
        'train_sharpe': best_config['train_sharpe'],
        'val_sharpe': best_config['val_sharpe'],
        'overfitting_score': best_config['overfitting_score']
    },
    'available_tickers': available_tickers
}

optimal_filename = f'optimal_config_{timestamp}.pkl'
with open(optimal_filename, 'wb') as f:
    pickle.dump(optimal_config, f)

print(f"✅ Configuración óptima guardada: {optimal_filename}")
print("=" * 80)

🎯 CONSTRUYENDO PORTFOLIO CON CONFIGURACIÓN ÓPTIMA
⚙️ Parámetros óptimos:
   pca_variance: 0.5
   umap_dim: 1
   umap_neighbors: 15
   umap_min_dist: 0.1
   n_cubes: 6
   perc_overlap: 0.4
   dbscan_eps: 0.4
   min_samples: 3

🔨 Construyendo mapper final...
   ✅ Grafo: 14 nodos, 7 enlaces
   📊 Clusters: 14 (tamaño promedio: 44.9)

✅ Visualización guardada: optimal_mapper_20251201_161158.html
✅ Configuración óptima guardada: optimal_config_20251201_161158.pkl
   ✅ Grafo: 14 nodos, 7 enlaces
   📊 Clusters: 14 (tamaño promedio: 44.9)

✅ Visualización guardada: optimal_mapper_20251201_161158.html
✅ Configuración óptima guardada: optimal_config_20251201_161158.pkl


## 📈 10.5 Construcción y Evaluación del Portfolio Óptimo

In [ ]:
# =======================================================
# 📊 CONSTRUIR PORTFOLIO COMPLETO CON MEJOR CONFIGURACIÓN
# =======================================================

print("📊 CONSTRUYENDO PORTFOLIO COMPLETO Y EVALUANDO PERFORMANCE")
print("=" * 80)

# Reconstruir el portfolio usando los mejores parámetros
# Seleccionar top clusters por Sharpe ratio
cluster_metrics = {}
for cluster_id, cluster_tickers in final_clusters.items():
    if len(cluster_tickers) < 2:
        continue
    
    cluster_returns = log_returns[cluster_tickers].mean(axis=1)
    cluster_sharpe = (cluster_returns.mean() * 252) / (cluster_returns.std() * np.sqrt(252))
    cluster_metrics[cluster_id] = {
        'sharpe': cluster_sharpe,
        'tickers': cluster_tickers,
        'size': len(cluster_tickers)
    }

# Seleccionar top 3 clusters
top_clusters_final = sorted(cluster_metrics.items(), 
                       key=lambda x: x[1]['sharpe'], 
                       reverse=True)[:3]

print(f"📊 Top 3 Clusters Seleccionados:")
for i, (cluster_id, cluster_info) in enumerate(top_clusters_final, 1):
    print(f"   {i}. Cluster {cluster_id}: {cluster_info['size']} tickers, Sharpe: {cluster_info['sharpe']:.3f}")

# Construir portafolio combinando top clusters
portfolio_weights_final = {}
cluster_weight = 1.0 / len(top_clusters_final)

for cluster_id, cluster_info in top_clusters_final:
    cluster_tickers = cluster_info['tickers']
    
    # Optimizar dentro del cluster (market cap weighted)
    intra_weights = optimize_cluster_portfolio(
        cluster_tickers, log_returns, valid_mcaps, method='market_cap'
    )
    
    # Agregar al portafolio global
    for ticker, weight in intra_weights.items():
        portfolio_weights_final[ticker] = portfolio_weights_final.get(ticker, 0) + weight * cluster_weight

# Normalizar pesos
total_weight = sum(portfolio_weights_final.values())
portfolio_weights_final = {t: w/total_weight for t, w in portfolio_weights_final.items()}

print(f"\n📊 Portfolio Final:")
print(f"   Número de tickers: {len(portfolio_weights_final)}")
print(f"   Top 5 pesos:")
sorted_weights = sorted(portfolio_weights_final.items(), key=lambda x: x[1], reverse=True)
for ticker, weight in sorted_weights[:5]:
    print(f"      {ticker}: {weight*100:.2f}%")

print("=" * 80)

### 10.6 Performance en Entrenamiento y Validación

In [ ]:
# =======================================================
# 📈 CALCULAR RETURNS DEL PORTFOLIO ÓPTIMO
# =======================================================

print("📈 CALCULANDO RETURNS DEL PORTFOLIO ÓPTIMO")
print("=" * 80)

# Returns en ENTRENAMIENTO
port_tickers_train = list(portfolio_weights_final.keys())
port_weights_train = np.array([portfolio_weights_final[t] for t in port_tickers_train])
portfolio_returns_train = (log_returns[port_tickers_train] * port_weights_train).sum(axis=1)

# Returns en VALIDACIÓN
val_tickers_available = [t for t in port_tickers_train if t in validation_returns.columns]
val_weights_adjusted = {t: portfolio_weights_final[t] for t in val_tickers_available}
total_w = sum(val_weights_adjusted.values())
val_weights_adjusted = {t: w/total_w for t, w in val_weights_adjusted.items()}

port_weights_val = np.array([val_weights_adjusted[t] for t in val_tickers_available])
portfolio_returns_val = (validation_returns[val_tickers_available] * port_weights_val).sum(axis=1)

# Returns del S&P 500 (SPY)
spy_returns_train = np.log(SPY_analysis / SPY_analysis.shift(1)).dropna()['SPY']
spy_returns_val = np.log(SPY_validation / SPY_validation.shift(1)).dropna()['SPY']

# Calcular métricas
portfolio_metrics_train = calculate_portfolio_metrics(log_returns, portfolio_returns_train, "TDA Portfolio - Train")
portfolio_metrics_val = calculate_portfolio_metrics(validation_returns, portfolio_returns_val, "TDA Portfolio - Val")
spy_metrics_train = calculate_portfolio_metrics(log_returns, spy_returns_train, "S&P 500 - Train")
spy_metrics_val = calculate_portfolio_metrics(validation_returns, spy_returns_val, "S&P 500 - Val")

# Mostrar comparación
print("\n📊 COMPARACIÓN DE PERFORMANCE:\n")
print("ENTRENAMIENTO (2022-01-01 a 2024-01-01):")
print(f"   TDA Portfolio:")
print(f"      Sharpe Ratio: {portfolio_metrics_train['sharpe']:.3f}")
print(f"      Retorno Anual: {portfolio_metrics_train['ann_return']*100:.2f}%")
print(f"      Volatilidad: {portfolio_metrics_train['ann_vol']*100:.2f}%")
print(f"      Max Drawdown: {portfolio_metrics_train['max_dd']*100:.2f}%")
print(f"\n   S&P 500 (SPY):")
print(f"      Sharpe Ratio: {spy_metrics_train['sharpe']:.3f}")
print(f"      Retorno Anual: {spy_metrics_train['ann_return']*100:.2f}%")
print(f"      Volatilidad: {spy_metrics_train['ann_vol']*100:.2f}%")
print(f"      Max Drawdown: {spy_metrics_train['max_dd']*100:.2f}%")

print("\n" + "-"*80)
print("\nVALIDACIÓN (2024-01-01 a 2025-11-01):")
print(f"   TDA Portfolio:")
print(f"      Sharpe Ratio: {portfolio_metrics_val['sharpe']:.3f}")
print(f"      Retorno Anual: {portfolio_metrics_val['ann_return']*100:.2f}%")
print(f"      Volatilidad: {portfolio_metrics_val['ann_vol']*100:.2f}%")
print(f"      Max Drawdown: {portfolio_metrics_val['max_dd']*100:.2f}%")
print(f"\n   S&P 500 (SPY):")
print(f"      Sharpe Ratio: {spy_metrics_val['sharpe']:.3f}")
print(f"      Retorno Anual: {spy_metrics_val['ann_return']*100:.2f}%")
print(f"      Volatilidad: {spy_metrics_val['ann_vol']*100:.2f}%")
print(f"      Max Drawdown: {spy_metrics_val['max_dd']*100:.2f}%")

print("\n" + "="*80)

### 10.7 Visualización: Retornos Acumulados (Portfolio vs S&P 500)

In [ ]:
# =======================================================
# 📊 GRÁFICA: RETORNOS ACUMULADOS - PORTFOLIO VS S&P 500
# =======================================================

print("📊 GENERANDO GRÁFICA: RETORNOS ACUMULADOS")
print("=" * 80)

# Calcular retornos acumulados
portfolio_cum_train = (1 + portfolio_returns_train).cumprod()
portfolio_cum_val = (1 + portfolio_returns_val).cumprod()
spy_cum_train = (1 + spy_returns_train).cumprod()
spy_cum_val = (1 + spy_returns_val).cumprod()

# Crear figura con subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Entrenamiento (2022-01-01 a 2024-01-01)', 
                   'Validación (2024-01-01 a 2025-11-01)'),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

# ENTRENAMIENTO
fig.add_trace(
    go.Scatter(
        x=portfolio_cum_train.index,
        y=portfolio_cum_train.values,
        name='TDA Portfolio',
        line=dict(color='#2E86AB', width=2.5),
        hovertemplate='<b>TDA Portfolio</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=spy_cum_train.index,
        y=spy_cum_train.values,
        name='S&P 500 (SPY)',
        line=dict(color='#A23B72', width=2.5, dash='dash'),
        hovertemplate='<b>S&P 500</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=1, col=1
)

# VALIDACIÓN
fig.add_trace(
    go.Scatter(
        x=portfolio_cum_val.index,
        y=portfolio_cum_val.values,
        name='TDA Portfolio (Val)',
        line=dict(color='#2E86AB', width=2.5),
        showlegend=False,
        hovertemplate='<b>TDA Portfolio</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(
        x=spy_cum_val.index,
        y=spy_cum_val.values,
        name='S&P 500 (Val)',
        line=dict(color='#A23B72', width=2.5, dash='dash'),
        showlegend=False,
        hovertemplate='<b>S&P 500</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=2, col=1
)

# Personalizar layout
fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_yaxes(title_text="Valor del Portfolio (Base = 1)", row=1, col=1)
fig.update_yaxes(title_text="Valor del Portfolio (Base = 1)", row=2, col=1)

fig.update_layout(
    title={
        'text': '📈 Retornos Acumulados: TDA Portfolio vs S&P 500',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#1f2937'}
    },
    height=800,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

print("✅ Gráfica generada")
print("=" * 80)

### 10.8 Visualización: Performance Individual de Tickers del Portfolio

In [ ]:
# =======================================================
# 📊 GRÁFICA: PERFORMANCE DE TICKERS INDIVIDUALES
# =======================================================

print("📊 GENERANDO GRÁFICA: PERFORMANCE DE TICKERS INDIVIDUALES")
print("=" * 80)

# Seleccionar top 10 tickers por peso
top_10_tickers = sorted_weights[:10]
top_10_tickers_list = [t[0] for t in top_10_tickers]

print(f"📊 Visualizando top 10 tickers por peso:")
for ticker, weight in top_10_tickers:
    print(f"   {ticker}: {weight*100:.2f}%")

# Calcular retornos acumulados para cada ticker
fig_tickers = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Entrenamiento: Top 10 Tickers del Portfolio', 
                   'Validación: Top 10 Tickers del Portfolio'),
    vertical_spacing=0.12
)

# Paleta de colores
colors = px.colors.qualitative.Set3[:10]

# ENTRENAMIENTO
for idx, ticker in enumerate(top_10_tickers_list):
    if ticker in log_returns.columns:
        ticker_cum_train = (1 + log_returns[ticker]).cumprod()
        
        fig_tickers.add_trace(
            go.Scatter(
                x=ticker_cum_train.index,
                y=ticker_cum_train.values,
                name=ticker,
                line=dict(color=colors[idx], width=1.5),
                hovertemplate=f'<b>{ticker}</b><br>Fecha: %{{x}}<br>Valor: %{{y:.3f}}<extra></extra>'
            ),
            row=1, col=1
        )

# VALIDACIÓN
for idx, ticker in enumerate(top_10_tickers_list):
    if ticker in validation_returns.columns:
        ticker_cum_val = (1 + validation_returns[ticker]).cumprod()
        
        fig_tickers.add_trace(
            go.Scatter(
                x=ticker_cum_val.index,
                y=ticker_cum_val.values,
                name=ticker,
                line=dict(color=colors[idx], width=1.5),
                showlegend=False,
                hovertemplate=f'<b>{ticker}</b><br>Fecha: %{{x}}<br>Valor: %{{y:.3f}}<extra></extra>'
            ),
            row=2, col=1
        )

# Agregar S&P 500 como referencia en ambos
fig_tickers.add_trace(
    go.Scatter(
        x=spy_cum_train.index,
        y=spy_cum_train.values,
        name='S&P 500',
        line=dict(color='black', width=2.5, dash='dash'),
        hovertemplate='<b>S&P 500</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=1, col=1
)

fig_tickers.add_trace(
    go.Scatter(
        x=spy_cum_val.index,
        y=spy_cum_val.values,
        name='S&P 500 (Val)',
        line=dict(color='black', width=2.5, dash='dash'),
        showlegend=False,
        hovertemplate='<b>S&P 500</b><br>Fecha: %{x}<br>Valor: %{y:.3f}<extra></extra>'
    ),
    row=2, col=1
)

# Personalizar layout
fig_tickers.update_xaxes(title_text="Fecha", row=2, col=1)
fig_tickers.update_yaxes(title_text="Valor (Base = 1)", row=1, col=1)
fig_tickers.update_yaxes(title_text="Valor (Base = 1)", row=2, col=1)

fig_tickers.update_layout(
    title={
        'text': '📊 Performance Individual: Top 10 Tickers del Portfolio vs S&P 500',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#1f2937'}
    },
    height=900,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.98,
        xanchor="right",
        x=0.99,
        bgcolor='rgba(255,255,255,0.8)'
    )
)

fig_tickers.show()

print("✅ Gráfica generada")
print("=" * 80)

### 10.9 Tabla Comparativa de Métricas

In [ ]:
# =======================================================
# 📊 TABLA COMPARATIVA DE MÉTRICAS
# =======================================================

print("📊 TABLA COMPARATIVA DE MÉTRICAS")
print("=" * 80)

# Crear DataFrame comparativo
comparison_data = {
    'Período': ['Entrenamiento', 'Entrenamiento', 'Validación', 'Validación'],
    'Estrategia': ['TDA Portfolio', 'S&P 500', 'TDA Portfolio', 'S&P 500'],
    'Sharpe Ratio': [
        portfolio_metrics_train['sharpe'],
        spy_metrics_train['sharpe'],
        portfolio_metrics_val['sharpe'],
        spy_metrics_val['sharpe']
    ],
    'Retorno Anual (%)': [
        portfolio_metrics_train['ann_return'] * 100,
        spy_metrics_train['ann_return'] * 100,
        portfolio_metrics_val['ann_return'] * 100,
        spy_metrics_val['ann_return'] * 100
    ],
    'Volatilidad (%)': [
        portfolio_metrics_train['ann_vol'] * 100,
        spy_metrics_train['ann_vol'] * 100,
        portfolio_metrics_val['ann_vol'] * 100,
        spy_metrics_val['ann_vol'] * 100
    ],
    'Max Drawdown (%)': [
        portfolio_metrics_train['max_dd'] * 100,
        spy_metrics_train['max_dd'] * 100,
        portfolio_metrics_val['max_dd'] * 100,
        spy_metrics_val['max_dd'] * 100
    ]
}

comparison_df = pd.DataFrame(comparison_data)

# Mostrar tabla
print("\n📊 TABLA COMPARATIVA:\n")
display(comparison_df.round(2))

# Calcular mejoras
print("\n📈 MEJORA DEL TDA PORTFOLIO vs S&P 500:\n")
print("ENTRENAMIENTO:")
sharpe_improvement_train = ((portfolio_metrics_train['sharpe'] - spy_metrics_train['sharpe']) / spy_metrics_train['sharpe']) * 100
return_improvement_train = portfolio_metrics_train['ann_return'] - spy_metrics_train['ann_return']
print(f"   Sharpe Ratio: {sharpe_improvement_train:+.1f}%")
print(f"   Retorno Anual: {return_improvement_train*100:+.2f} puntos porcentuales")

print("\nVALIDACIÓN:")
sharpe_improvement_val = ((portfolio_metrics_val['sharpe'] - spy_metrics_val['sharpe']) / spy_metrics_val['sharpe']) * 100
return_improvement_val = portfolio_metrics_val['ann_return'] - spy_metrics_val['ann_return']
print(f"   Sharpe Ratio: {sharpe_improvement_val:+.1f}%")
print(f"   Retorno Anual: {return_improvement_val*100:+.2f} puntos porcentuales")

print("\n" + "=" * 80)

## 📈 11. Resumen Final y Recomendaciones

In [ ]:
# =======================================================
# 📈 RESUMEN FINAL
# =======================================================

print("📈 RESUMEN FINAL DE OPTIMIZACIÓN")
print("=" * 80)

print(f"\n🔬 Experimentos Realizados:")
print(f"   Total de configuraciones probadas: {len(results)}")
print(f"   Configuraciones fallidas: {len(failed_combinations)}")
print(f"   Tasa de éxito: {len(results)/(len(results)+len(failed_combinations))*100:.1f}%")

print(f"\n🏆 Mejor Configuración:")
print(f"   Sharpe Validación: {best_config['val_sharpe']:.3f}")
print(f"   Sharpe Entrenamiento: {best_config['train_sharpe']:.3f}")
print(f"   Overfitting Score: {best_config['overfitting_score']:.3f}")

print(f"\n📊 Parámetros Más Importantes (Top 3):")
for i, (param, corr) in enumerate(sorted_importance[:3], 1):
    print(f"   {i}. {param}: {corr:+.3f}")

print(f"\n📁 Archivos Generados:")
print(f"   • {csv_output}")
print(f"   • {pickle_output}")
print(f"   • {html_filename}")
print(f"   • {optimal_filename}")

print(f"\n💡 Recomendaciones:")
print(f"   1. Usar los parámetros de 'best_config' para producción")
print(f"   2. Revisar visualización HTML para entender estructura del mapper")
print(f"   3. Considerar re-entrenamiento cada 3-6 meses")
print(f"   4. Monitorear métricas de validación en tiempo real")

print("\n" + "=" * 80)
print("✅ OPTIMIZACIÓN COMPLETADA")
print("=" * 80)

📈 RESUMEN FINAL DE OPTIMIZACIÓN

🔬 Experimentos Realizados:
   Total de configuraciones probadas: 108
   Configuraciones fallidas: 0
   Tasa de éxito: 100.0%

🏆 Mejor Configuración:
   Sharpe Validación: 1.468
   Sharpe Entrenamiento: 1.475
   Overfitting Score: 0.005

📊 Parámetros Más Importantes (Top 3):
   1. pca_variance: -0.116
   2. umap_dim: +0.096
   3. umap_neighbors: +nan

📁 Archivos Generados:
   • optimization_results_20251201_161158.csv
   • optimization_full_20251201_161158.pkl
   • optimal_mapper_20251201_161158.html
   • optimal_config_20251201_161158.pkl

💡 Recomendaciones:
   1. Usar los parámetros de 'best_config' para producción
   2. Revisar visualización HTML para entender estructura del mapper
   3. Considerar re-entrenamiento cada 3-6 meses
   4. Monitorear métricas de validación en tiempo real

✅ OPTIMIZACIÓN COMPLETADA
